# NB02: MAG Functional Annotation via CTS

Submit QC-passing MAGs from NB01 to CTS for bakta annotation,
then parse results into a KO profile table matching the cultured cohort format.

**Requires**: NB01 outputs (per-bin FASTAs in `data/mag_fastas/`),
CTS access (`/cts` skill)

In [ ]:
import os, json, glob
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('../data')
MAG_DIR = DATA_DIR / 'mag_fastas'
ANNOT_DIR = DATA_DIR / 'mag_annotations'
ANNOT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Inventory MAG FASTAs from NB01

In [ ]:
mag_meta = pd.read_csv(DATA_DIR / 'mag_metadata.tsv', sep='\t')
fasta_files = sorted(MAG_DIR.glob('*.fasta'))

print(f'{len(fasta_files)} MAG FASTAs available')
print(f'{len(mag_meta)} MAGs in metadata')

total_size_mb = sum(f.stat().st_size for f in fasta_files) / 1e6
print(f'Total FASTA size: {total_size_mb:.0f} MB')

## 2. Submit bakta Jobs to CTS

Use the `/cts` skill to submit batch bakta annotation.
This cell documents the submission — actual submission is done via the CLI.

In [ ]:
# CTS submission manifest
# Each MAG FASTA gets a bakta job
# bakta outputs: .tsv (annotations), .gff3, .gbff, .faa (proteins), .hypotheticals.tsv

manifest = []
for fasta in fasta_files:
    manifest.append({
        'bin_name': fasta.stem,
        'fasta_path': str(fasta),
        'fasta_size_mb': fasta.stat().st_size / 1e6
    })

manifest_df = pd.DataFrame(manifest)
manifest_df.to_csv(DATA_DIR / 'cts_bakta_manifest.tsv', sep='\t', index=False)
print(f'Manifest: {len(manifest_df)} jobs')
print(f'Size range: {manifest_df["fasta_size_mb"].min():.1f} - {manifest_df["fasta_size_mb"].max():.1f} MB')
print(f'\nSubmit via: /cts submit bakta --manifest data/cts_bakta_manifest.tsv')

## 3. Parse bakta Outputs → KO Profiles

In [ ]:
# After CTS jobs complete, bakta .tsv files land in ANNOT_DIR
# Parse them into the same format as cultured_ko_profiles.tsv:
#   genome_id, ko_id, ko_description, n_copies

annot_files = sorted(ANNOT_DIR.glob('*.tsv'))
if not annot_files:
    print('No bakta annotation files found yet.')
    print('Run CTS bakta jobs first, then re-run this cell.')
else:
    all_ko_rows = []
    for af in annot_files:
        bin_name = af.stem
        try:
            annot = pd.read_csv(af, sep='\t', comment='#')
        except Exception as e:
            print(f'Error reading {af}: {e}')
            continue
        
        # bakta output columns vary; look for KEGG annotations
        # Typically in 'db_xrefs' column as 'KEGG:K00001' entries
        if 'db_xrefs' not in annot.columns:
            print(f'  {bin_name}: no db_xrefs column')
            continue
        
        ko_counts = {}
        for _, row in annot.iterrows():
            if pd.isna(row['db_xrefs']):
                continue
            for xref in str(row['db_xrefs']).split(','):
                xref = xref.strip()
                if xref.startswith('KEGG:K'):
                    ko_id = xref.replace('KEGG:', '')
                    ko_counts[ko_id] = ko_counts.get(ko_id, 0) + 1
        
        for ko_id, n in ko_counts.items():
            all_ko_rows.append({
                'genome_id': bin_name,
                'ko_id': ko_id,
                'ko_description': '',
                'n_copies': n
            })
    
    mag_ko_df = pd.DataFrame(all_ko_rows)
    print(f'{len(mag_ko_df)} MAG-KO pairs')
    print(f'{mag_ko_df["genome_id"].nunique()} MAGs with KO annotations')
    print(f'{mag_ko_df["ko_id"].nunique()} unique KOs')

## 4. Backfill KO Descriptions

In [ ]:
# Use the cultured KO description table for backfill
if len(annot_files) > 0 and len(mag_ko_df) > 0:
    cultured_ko = pd.read_csv(DATA_DIR / 'cultured_ko_profiles.tsv', sep='\t',
                               usecols=['ko_id', 'ko_description']).drop_duplicates('ko_id')
    ko_desc_map = dict(zip(cultured_ko['ko_id'], cultured_ko['ko_description']))
    
    mag_ko_df['ko_description'] = mag_ko_df['ko_id'].map(ko_desc_map).fillna('')
    
    mag_ko_df.to_csv(DATA_DIR / 'mag_ko_profiles.tsv', sep='\t', index=False)
    print(f'Saved {len(mag_ko_df)} MAG-KO pairs to data/mag_ko_profiles.tsv')
    print(f'KOs with descriptions: {(mag_ko_df["ko_description"] != "").sum()}/{len(mag_ko_df)}')

## 5. MAG KO Summary Statistics

In [ ]:
if len(annot_files) > 0 and len(mag_ko_df) > 0:
    kos_per_mag = mag_ko_df.groupby('genome_id')['ko_id'].nunique()
    print(f'KOs per MAG:')
    print(f'  mean={kos_per_mag.mean():.0f}, median={kos_per_mag.median():.0f}')
    print(f'  min={kos_per_mag.min()}, max={kos_per_mag.max()}')
    print(f'  Q25={kos_per_mag.quantile(0.25):.0f}, Q75={kos_per_mag.quantile(0.75):.0f}')
    
    # Compare with cultured
    cultured_ko_full = pd.read_csv(DATA_DIR / 'cultured_ko_profiles.tsv', sep='\t')
    cultured_kpg = cultured_ko_full.groupby('genome_id')['ko_id'].nunique()
    print(f'\nComparison with cultured genomes:')
    print(f'  Cultured: mean={cultured_kpg.mean():.0f} KOs/genome')
    print(f'  MAGs:     mean={kos_per_mag.mean():.0f} KOs/genome')
else:
    print('Waiting for bakta annotation results.')